**HuggingFace authentification**

In [12]:
from huggingface_hub import notebook_login
notebook_login()

**Installing Git Large File Storage (LFS)**

In [2]:
 !apt-get install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.2).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.


**Install and load libraries**

In [3]:
# Install the evaluate library
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.7/472.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.0 MB/s eta 0:00:00


In [4]:
# Importing libraries
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSequenceClassification

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# Load dataset
ds = load_dataset('imdb')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
# Tokenize the data
def tokenize_data(examples):
    return tokenizer(examples['text'], padding = True, truncation = True)

tokenized_ds = ds.map(tokenize_data, batched = True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [9]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased-finetuned-sst-2-english')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [10]:
# Setup evaluation
metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

**Train the model**

In [13]:
from transformers import TrainingArguments, Trainer
repo_name = 'lhl-sentiment-analysis-model'

training_args = TrainingArguments(
    output_dir=repo_name,
    learning_rate=3e-5,             # Increased from 2e-5 to 3e-5
    per_device_train_batch_size=32, # Increased from 16 to 32
    per_device_eval_batch_size=32,  # Increased from 16 to 32
    num_train_epochs=3,             # Increased from 1 to 3
    push_to_hub=True,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss
500,0.240300
1000,0.167300
1500,0.115500
2000,0.065000


TrainOutput(global_step=2346, training_loss=0.13431114268323244, metrics={'train_runtime': 3476.6691, 'train_samples_per_second': 21.572, 'train_steps_per_second': 0.675, 'total_flos': 9935054899200000.0, 'train_loss': 0.13431114268323244, 'epoch': 3.0})

In [14]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.2608377933502197, 'eval_f1': 0.9358984593278519, 'eval_runtime': 420.3653, 'eval_samples_per_second': 59.472, 'eval_steps_per_second': 1.86, 'epoch': 3.0}


**Evaluate the model**

**Results analysis**

After tuning the model's hyperparameters:

**Learning Rate:** Increased from 2e-5 to 3e-5.

**Batch Size:** Increased from 16 to 32 for both training and evaluation.

**Number of Training Epochs:** Increased from 1 to 3.


The results after tuning are as follows:

**Evaluation Loss:** Changed from 0.1861 to 0.2608, indicating a slight increase in loss after further training.

**F1 Score:** Improved from 0.9336 to 0.9359, demonstrating enhanced performance in balancing precision and recall.

**Evaluation Runtime:** Increased from 411.40 seconds to 420.37 seconds.

**Samples Per Second:** Decreased from 60.77 to 59.47, indicating a slight drop in throughput.

**Steps Per Second:** Decreased from 3.80 to 1.86.

Overall, the tuning process resulted in a marginal increase in the F1 score, suggesting improved model performance in sentiment classification, despite a slight rise in loss and decreased processing speed.

**Push model to the hub**

In [15]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/wjura/lhl-sentiment-analysis-model/commit/904999774b675931f1d854c34da454acc6d1f61e', commit_message='End of training', commit_description='', oid='904999774b675931f1d854c34da454acc6d1f61e', pr_url=None, pr_revision=None, pr_num=None)